# पाठ 09 - मेटाकॉग्निशन डिज़ाइन पैटर्न


## सेटअप

यह नोटबुक Microsoft Agent Framework का उपयोग करके Metacognition डिज़ाइन पैटर्न को प्रदर्शित करता है।

**पूर्वापेक्षाएँ:**
- Azure OpenAI डिप्लॉयमेंट पर्यावरण चर के माध्यम से कॉन्फ़िगर किया गया
- Azure CLI प्रमाणित (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## मेटाकॉग्निशन क्या है?

मेटाकॉग्निशन का मतलब है **सोच के बारे में सोचना**। AI एजेंट्स के संदर्भ में, इसका मतलब ऐसे एजेंट बनाना है जो कि:

- अपनी आउटपुट और तर्क प्रक्रिया पर **स्वयं-प्रतिबिंबित** करें
- **त्रुटियों का पता लगाएं** और चुपचाप विफल होने की बजाय सहजहीनता से ठीक हो जाएं
- मूल्यांकन करें कि उनके उत्तर पूर्ण और सहायक हैं या नहीं
- जब प्रारंभिक तरीका काम नहीं करता है तो अपनी रणनीति में **अनुकूलन** करें (जैसे, बैकअप सिस्टम पर वापस जाना)

एक मेटाकॉग्निटिव एजेंट सिर्फ प्रश्नों का उत्तर नहीं देता — वह अपनी स्वयं की प्रदर्शन की निगरानी करता है और त्वरित समायोजन करता है।


## प्राथमिक और बैकअप उपकरण

एक सामान्य मेटाकॉग्निशन पैटर्न है **फालबैक रणनीति**। एजेंट पहले एक प्राथमिक उपकरण आज़माता है; यदि यह विफल रहता है (जैसे, 404 त्रुटि), तो एजेंट विफलता को पहचानता है और पारदर्शी रूप से बैकअप उपकरण पर स्विच करता है।

यह वास्तविक दुनिया की प्रणालियों को दर्शाता है जहाँ प्राथमिक सेवाएँ अनुपलब्ध हो सकती हैं और एजेंट को समस्या का स्व-निदान करना होता है इससे पहले कि वे वैकल्पिक मार्ग चुनें।

नीचे हम दो उड़ान खोज उपकरण परिभाषित करते हैं:
- **प्राथमिक** — पेरिस, टोक्यो, और बार्सिलोना को कवर करता है
- **बैकअप** — बर्लिन, सिडनी, और न्यू यॉर्क सिटी को कवर करता है


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## त्रुटि पुनर्प्राप्ति के साथ आत्म-परावर्तक एजेंट

नीचे दिए गए एजेंट को प्राथमिक उड़ान प्रणाली को पहले प्रयास करने, असफलताओं को पहचानने, और पारदर्शी ढंग से बैकअप सिस्टम पर वापस जाने का निर्देश दिया गया है। प्रत्येक प्रतिक्रिया के बाद यह संक्षेप में स्वयं मूल्यांकन करता है कि क्या उसने उपयोगकर्ता के प्रश्न का पूरी तरह से उत्तर दिया है।


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## आत्म-मूल्यांकन पैटर्न

मेटाकॉग्निशन का एक अन्य पहलू **आत्म-मूल्यांकन** है: एक अलग एजेंट (या दूसरे पास में वही एजेंट) प्रतिक्रिया की पूर्णता, सटीकता और उपयोगिता की समीक्षा करता है।

नीचे हम एक `ResponseEvaluator` एजेंट बनाते हैं जो यात्रा-एजेंट की प्रतिक्रियाओं को तीन आयामों पर स्कोर करता है।


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## सारांश

इस पाठ में आपने सीखा कि Microsoft Agent Framework का उपयोग करके **मेटाकॉग्निटिव एजेंट्स** कैसे बनाएं:

- **स्व-परावर्तन**: ऐसे एजेंट जो अपनी सोच पर नजर रखते हैं और पारदर्शी रूप से बताते हैं कि क्या हुआ।
- **Fallbacks के साथ त्रुटि पुनर्प्राप्ति**: एक प्राथमिक + बैकअप टूल पैटर्न जहां एजेंट विफलताओं (जैसे 404 त्रुटियां) का पता लगाता है और स्वचालित रूप से एक वैकल्पिक स्रोत आजमाता है।
- **स्व-मूल्यांकन**: एक अलग मूल्यांकन एजेंट जो प्रतिक्रियाओं को पूर्णता, सटीकता, और सहायकता के लिए स्कोर करता है।

ये पैटर्न एजेंट्स को अधिक मजबूत, पारदर्शी, और भरोसेमंद बनाते हैं — जो उत्पादन तैनाती के लिए महत्वपूर्ण गुण हैं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
